# Cam And Motor Check

This notebook combines the camera check and motor control steps into one browser page.

Use it before data collection so we can verify everything from a single place:

- camera opens cleanly
- lane mask updates in real time
- serial control responds correctly
- stop control is always available
            


In [ ]:
from pathlib import Path
import glob
import os
import socket
import sys

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

video_devices = sorted(glob.glob('/dev/video*'))
serial_candidates = [p for p in ('/dev/ttyTHS1', '/dev/ttyTHS2') if os.path.exists(p)]

hostname = socket.gethostname()
ip_addresses = sorted({addr[-1][0] for addr in socket.getaddrinfo(hostname, None, family=socket.AF_INET)})

print('Project root:', project_root)
print('Video devices:', video_devices or ['none found'])
print('Serial candidates:', serial_candidates or ['none found'])
print('Host IPv4:', ip_addresses or ['127.0.0.1'])
            


In [ ]:
import os
import shlex
import signal
import subprocess
import threading
import time

import ipywidgets as widgets
from IPython.display import HTML, IFrame, display

server_process = None
log_thread = None
stop_logs = threading.Event()


def python_executable() -> str:
    candidate = project_root / '.venv' / 'bin' / 'python'
    return str(candidate) if candidate.exists() else sys.executable


def build_url(host: str, port: int) -> str:
    browser_host = '127.0.0.1' if host == '0.0.0.0' else host
    return f'http://{browser_host}:{port}'


def build_command() -> list[str]:
    return [
        python_executable(),
        str(project_root / 'scripts' / 'teleop_server.py'),
        '--host', host_text.value,
        '--http-port', str(http_port.value),
        '--camera-source', camera_source.value,
        '--sensor-id', str(sensor_id.value),
        '--device-index', str(device_index.value),
        '--width', str(frame_width.value),
        '--height', str(frame_height.value),
        '--warmup-frames', str(warmup_frames.value),
        '--port', serial_port.value,
        '--baudrate', str(baudrate.value),
    ]


def append_log(text: str) -> None:
    with log_output:
        print(text, end='')


def pump_logs(process: subprocess.Popen) -> None:
    while not stop_logs.is_set():
        line = process.stdout.readline()
        if not line:
            break
        append_log(line)


def refresh_preview(*_args) -> None:
    url = build_url(host_text.value, http_port.value)
    launch_url.value = f'<b>Teleop URL:</b> <a href="{url}" target="_blank">{url}</a>'
    preview_output.clear_output(wait=True)
    with preview_output:
        display(IFrame(src=url, width='100%', height=960))


def refresh_command(*_args) -> None:
    command_preview.value = '<b>Launch command:</b><br><code>' + ' '.join(shlex.quote(part) for part in build_command()) + '</code>'
    refresh_preview()


def start_server(_button=None) -> None:
    global server_process, log_thread
    stop_server()
    cmd = build_command()
    stop_logs.clear()
    server_process = subprocess.Popen(
        cmd,
        cwd=project_root,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    status.value = f'<b>Status:</b> launching PID {server_process.pid}'
    log_thread = threading.Thread(target=pump_logs, args=(server_process,), daemon=True)
    log_thread.start()
    time.sleep(1.2)
    if server_process.poll() is None:
        status.value = f'<b>Status:</b> running PID {server_process.pid}'
    else:
        status.value = f'<b>Status:</b> exited with code {server_process.returncode}'
    refresh_preview()


def stop_server(_button=None) -> None:
    global server_process
    if server_process is None:
        return
    stop_logs.set()
    if server_process.poll() is None:
        server_process.send_signal(signal.SIGINT)
        try:
            server_process.wait(timeout=3)
        except subprocess.TimeoutExpired:
            server_process.terminate()
            server_process.wait(timeout=3)
    status.value = '<b>Status:</b> stopped'
    server_process = None


host_text = widgets.Text(value='0.0.0.0', description='Host')
http_port = widgets.IntText(value=8765, description='HTTP Port')
camera_source = widgets.Dropdown(options=['usb', 'csi'], value='usb', description='Camera')
sensor_id = widgets.IntText(value=0, description='Sensor')
device_index = widgets.IntText(value=0, description='Device')
frame_width = widgets.IntText(value=1280, description='Width')
frame_height = widgets.IntText(value=720, description='Height')
warmup_frames = widgets.IntText(value=12, description='Warmup')
serial_port = widgets.Text(value=serial_candidates[0] if serial_candidates else '/dev/ttyTHS1', description='Serial')
baudrate = widgets.IntText(value=115200, description='Baud')

start_button = widgets.Button(description='Start Teleop', button_style='success')
stop_button = widgets.Button(description='Stop Teleop', button_style='danger')
refresh_button = widgets.Button(description='Refresh Preview')
status = widgets.HTML(value='<b>Status:</b> idle')
launch_url = widgets.HTML()
command_preview = widgets.HTML()
log_output = widgets.Output(layout={'border': '1px solid #bbb', 'max_height': '280px', 'overflow_y': 'auto'})
preview_output = widgets.Output()

for widget in [host_text, http_port, camera_source, sensor_id, device_index, frame_width, frame_height, warmup_frames, serial_port, baudrate]:
    widget.observe(refresh_command, names='value')

start_button.on_click(start_server)
stop_button.on_click(stop_server)
refresh_button.on_click(refresh_preview)

controls = widgets.VBox([
    widgets.HBox([host_text, http_port]),
    widgets.HBox([camera_source, sensor_id, device_index]),
    widgets.HBox([frame_width, frame_height, warmup_frames]),
    widgets.HBox([serial_port, baudrate]),
    widgets.HBox([start_button, stop_button, refresh_button]),
    status,
    launch_url,
    command_preview,
])

display(controls)
display(log_output)
display(preview_output)
refresh_command()
            


## Notes

- If the page loads but the camera panel is black, wait a second for the camera warmup.
- If serial control says unavailable, check UART wiring and `dialout` access, then restart the teleop server from this notebook.
- `0.0.0.0` is the right bind address for the server. The notebook preview opens `127.0.0.1` locally, and you can also open the printed URL from another browser on the same network.
            
